In [3]:
import json
import re
from pathlib import Path

# ===== CONFIG =====
JSON_FILE = "/home/name-1/AI-Agent/new-born/code/financial_lines_psm6.json"  # path to your JSON file  # Path to your OCR JSON
KEYWORDS = {
    "balance_sheet": ["balance sheet", "statement of financial position"],
    "income_statement": ["income statement", "statement of profit", "comprehensive income"]
}
STOP_WORDS = {
    "balance_sheet": ["income statement", "profit", "comprehensive income", "cash flow"],
    "income_statement": ["cash flow", "equity", "notes"]
}

# ===== STEP 1: Read JSON =====
print(f"Loading JSON from: {JSON_FILE}")
data = json.loads(Path(JSON_FILE).read_text(encoding="utf-8"))

# ===== STEP 2: Flatten JSON =====
def flatten_json(obj):
    texts = []
    if isinstance(obj, dict):
        for v in obj.values():
            texts.extend(flatten_json(v))
    elif isinstance(obj, list):
        for item in obj:
            texts.extend(flatten_json(item))
    elif isinstance(obj, str):
        texts.append(obj.strip())
    return texts

all_texts = flatten_json(data)

print("\n=== SAMPLE RAW TEXTS FROM JSON ===")
for i, t in enumerate(all_texts[:40], start=1):
    print(f"{i:03}: {t}")
print(f"\nTotal text entries in JSON: {len(all_texts)}")

# ===== STEP 3: Extract section =====
def extract_section(texts, start_keywords, stop_keywords):
    start_idx = None
    stop_idx = None
    for i, t in enumerate(texts):
        lower_t = t.lower()
        if start_idx is None and any(k in lower_t for k in start_keywords):
            start_idx = i
        elif start_idx is not None and any(k in lower_t for k in stop_keywords):
            stop_idx = i
            break
    if start_idx is not None:
        return texts[start_idx:stop_idx] if stop_idx else texts[start_idx:]
    return []

# ===== STEP 4: Detect table-like lines =====
def is_table_row(line):
    """Returns True if line has at least one number, suggesting it's a table row."""
    # Match numbers like 1,234 or 1234 or (123) or 1.23
    return bool(re.search(r"\d[\d,().]*", line))

def clean_lines(lines):
    """Remove empty lines and non-table-like lines"""
    cleaned = []
    for line in lines:
        line = re.sub(r"\s+", " ", line).strip()  # normalize spaces
        if not line:
            continue
        if is_table_row(line) or any(len(line.split()) > 1 for _ in [0]):  # keep if has numbers or multiple words
            cleaned.append(line)
    return cleaned

# ===== STEP 5: Process both sections =====
print("\n=== EXPECTED KEYWORDS ===")
for sec, keys in KEYWORDS.items():
    print(f"{sec}: {keys}")

balance_raw = extract_section(all_texts, KEYWORDS["balance_sheet"], STOP_WORDS["balance_sheet"])
income_raw = extract_section(all_texts, KEYWORDS["income_statement"], STOP_WORDS["income_statement"])

balance_clean = clean_lines(balance_raw)
income_clean = clean_lines(income_raw)

# ===== STEP 6: Output =====
print("\n=== BALANCE SHEET RAW (first 20 lines) ===")
for i, line in enumerate(balance_raw[:20], start=1):
    print(f"{i:03}: {line}")

print("\n=== BALANCE SHEET CLEAN (table-like) ===")
for i, line in enumerate(balance_clean, start=1):
    print(f"{i:03}: {line}")

print("\n=== INCOME STATEMENT RAW (first 20 lines) ===")
for i, line in enumerate(income_raw[:20], start=1):
    print(f"{i:03}: {line}")

print("\n=== INCOME STATEMENT CLEAN (table-like) ===")
for i, line in enumerate(income_clean, start=1):
    print(f"{i:03}: {line}")

# ===== STEP 7: Save results =====
Path("balance_sheet_clean.txt").write_text("\n".join(balance_clean), encoding="utf-8")
Path("income_statement_clean.txt").write_text("\n".join(income_clean), encoding="utf-8")

print("\nCleaned table-like sections saved to:")
print("  balance_sheet_clean.txt")
print("  income_statement_clean.txt")


Loading JSON from: /home/name-1/AI-Agent/new-born/code/financial_lines_psm6.json

=== SAMPLE RAW TEXTS FROM JSON ===
001: 7 Addisu Addis Bole Sub Ababa, Alemu -city, Ethiopia Certified Wereda 02 Audit Firm
002: )
003: , HABESHA PETROLEUM AND PETROLEUM PRODUCTS
004: DISTRIBUTER PRIVATE LIMITED COMPANY
005: INDEPENDENT AUDITORS' REPORT AND FINANCIAL
006: STATEMENTS
007: FOR THE YEAR ENDED 30 JUNE 2024
008: ] HABESHA PETROLEUM AND PETROLEUM PRODUCTS DISTRIBUTER PLC
009: INDEPENDENT AUDITORS' REPORT AND FINANCIAL STATEMENTS
010: FOR THE YEAR ENDED 30 JUNE 2024
011: COMPANY GENERAL INFORMATION
012: MANAGEMENT Ato Abatneh Chanie General Manager
013: Ato Abebe Beza Finance Manager
014: REGISTERED OFFICE Habesha Petroleum and Petroleum Products Distributor PLC
015: Kirkos Sub-city, Woreda 02
016: Addis Ababa
017: Ethiopia
018: AUDITORS Addisu Alemu Cerified Audit Firm
019: Bole Sub -city, Wereda 02
020: Addis Ababa,
021: Ethiopia
022: BANKERS Commercial Bank of Ethiopia Abyssinia Bank
023: Add

In [12]:
import json
import re
from pathlib import Path

# ===== CONFIG =====
JSON_FILE = "/home/name-1/AI-Agent/new-born/code/financial_lines_psm6.json"  # Path to your OCR JSON
KEYWORDS = {
    "balance_sheet": ["balance sheet", "statement of financial position"],
    "income_statement": ["income statement", "statement of profit", "comprehensive income"]
}
STOP_WORDS = {
    "balance_sheet": ["income statement", "profit", "comprehensive income", "cash flow"],
    "income_statement": ["cash flow", "equity", "notes"]
}

# ===== STEP 1: Read JSON =====
print(f"Loading JSON from: {JSON_FILE}")
data = json.loads(Path(JSON_FILE).read_text(encoding="utf-8"))

# ===== STEP 2: Flatten JSON =====
def flatten_json(obj):
    texts = []
    if isinstance(obj, dict):
        for v in obj.values():
            texts.extend(flatten_json(v))
    elif isinstance(obj, list):
        for item in obj:
            texts.extend(flatten_json(item))
    elif isinstance(obj, str):
        texts.append(obj.strip())
    return texts

all_texts = flatten_json(data)
print(f"\nTotal text entries in JSON: {len(all_texts)}")

# ===== STEP 3: Extract section =====
def extract_section(texts, start_keywords, stop_keywords):
    start_idx = None
    stop_idx = None
    for i, t in enumerate(texts):
        lower_t = t.lower()
        if start_idx is None and any(k in lower_t for k in start_keywords):
            start_idx = i
        elif start_idx is not None and any(k in lower_t for k in stop_keywords):
            stop_idx = i
            break
    if start_idx is not None:
        return texts[start_idx:stop_idx] if stop_idx else texts[start_idx:]
    return []

# ===== STEP 4: Detect table-like lines =====
def is_table_row(line):
    return bool(re.search(r"\d[\d,().-]*", line))

def clean_lines(lines):
    cleaned = []
    for line in lines:
        line = re.sub(r"\s+", " ", line).strip()
        if not line:
            continue
        if is_table_row(line) or len(line.split()) > 1:
            cleaned.append(line)
    return cleaned

# ===== STEP 5: Parse table row =====
def parse_table_row(line):
    parts = line.rsplit(" ", 3)
    if len(parts) < 3:
        return None

    item_note = parts[0].rsplit(" ", 1)
    if len(item_note) == 2 and re.match(r"[\d\.]+", item_note[1]):
        item, note = item_note
    else:
        item, note = parts[0], None

    current, previous = parts[1], parts[2]

    def parse_number(s):
        s = s.replace(",", "").replace("-", "0").replace("(", "-").replace(")", "")
        try:
            return float(s)
        except:
            return 0.0

    return {
        "item": item.rstrip("-").strip(),
        "note": note.strip() if note else None,
        "current_year": parse_number(current),
        "previous_year": parse_number(previous)
    }

def process_section(section_lines):
    cleaned = clean_lines(section_lines)
    rows = []
    for line in cleaned:
        row = parse_table_row(line)
        if row:
            rows.append(row)
    return rows

# ===== STEP 6: Extract and process sections =====
balance_raw = extract_section(all_texts, KEYWORDS["balance_sheet"], STOP_WORDS["balance_sheet"])
income_raw = extract_section(all_texts, KEYWORDS["income_statement"], STOP_WORDS["income_statement"])

balance_data = process_section(balance_raw)
income_data = process_section(income_raw)

# ===== STEP 7: Save JSON =====
output_json = {
    "BALANCE_SHEET": balance_data,
    "INCOME_STATEMENT": income_data
}

output_file = "financial_statements_parsed.json"
Path(output_file).write_text(json.dumps(output_json, indent=4), encoding="utf-8")

print(f"\nParsed financial statements saved to: {output_file}")
print(f"Balance sheet rows: {len(balance_data)}, Income statement rows: {len(income_data)}")


Loading JSON from: /home/name-1/AI-Agent/new-born/code/financial_lines_psm6.json

Total text entries in JSON: 1495

Parsed financial statements saved to: financial_statements_parsed.json
Balance sheet rows: 1, Income statement rows: 1


In [13]:
import json
import re
from pathlib import Path

# ===== CONFIG =====
JSON_FILE = "/home/name-1/AI-Agent/new-born/code/financial_lines_psm6.json"

KEYWORDS = {
    "BALANCE_SHEET": ["balance sheet", "statement of financial position"],
    "INCOME_STATEMENT": ["income statement", "statement of profit", "comprehensive income"]
}

STOP_WORDS = {
    "BALANCE_SHEET": ["income statement", "profit", "comprehensive income", "cash flow"],
    "INCOME_STATEMENT": ["cash flow", "equity", "notes"]
}

# ===== STEP 1: Load JSON =====
data = json.loads(Path(JSON_FILE).read_text(encoding="utf-8"))

# ===== STEP 2: Flatten JSON safely =====
def flatten_json(obj):
    texts = []
    if isinstance(obj, dict):
        for v in obj.values():
            texts.extend(flatten_json(v))
    elif isinstance(obj, list):
        for item in obj:
            texts.extend(flatten_json(item))
    elif isinstance(obj, str):
        texts.append(obj.strip())
    return texts

all_texts = flatten_json(data)

# ===== STEP 3: Clean text =====
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"[^\x20-\x7E]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# ===== STEP 4: Extract section =====
def extract_section(texts, start_keywords, stop_keywords):
    start_idx = None
    stop_idx = None
    for i, t in enumerate(texts):
        lower_t = t.lower()
        if start_idx is None and any(k in lower_t for k in start_keywords):
            start_idx = i
        elif start_idx is not None and any(k in lower_t for k in stop_keywords):
            stop_idx = i
            break
    if start_idx is not None:
        return texts[start_idx:stop_idx] if stop_idx else texts[start_idx:]
    return []

# ===== STEP 5: Parse table row =====
def parse_table_row(line):
    parts = line.split()
    if len(parts) < 2:
        return None

    # Find numeric parts
    numbers = [p.replace(",", "").replace("(", "-").replace(")", "") for p in parts if re.search(r"[\d\-]", p)]
    
    # Replace "-" with 0
    numbers = [0 if n.strip() == "-" else float(n) for n in numbers]

    # Extract item and note
    note = None
    item_parts = []
    for p in parts:
        if re.match(r"^\d+(\.\d+)?$", p):
            break
        item_parts.append(p)
    item_str = " ".join(item_parts).strip()
    if " " in item_str:
        # Optional: split note if last part is like 12.1
        if re.match(r"^\d+(\.\d+)?$", item_parts[-1]):
            note = item_parts[-1]
            item_str = " ".join(item_parts[:-1])

    current = numbers[0] if len(numbers) > 0 else 0
    previous = numbers[1] if len(numbers) > 1 else 0

    return {
        "item": item_str,
        "note": note,
        "current_year": current,
        "previous_year": previous
    }

# ===== STEP 6: Extract table rows =====
def extract_table_rows(lines):
    rows = []
    for line in lines:
        line = clean_text(line)
        if not line:
            continue
        row = parse_table_row(line)
        if row:
            rows.append(row)
    return rows

# ===== STEP 7: Process sections =====
results = {}
for section, keywords in KEYWORDS.items():
    raw_section = extract_section(all_texts, keywords, STOP_WORDS[section])
    table_rows = extract_table_rows(raw_section)
    results[section] = table_rows

# ===== STEP 8: Save results to JSON =====
output_file = "financial_statements.json"
Path(output_file).write_text(json.dumps(results, indent=4), encoding="utf-8")

print(f"Financial statements extracted and saved to {output_file}")


Financial statements extracted and saved to financial_statements.json


In [5]:
import json
import re

# ==== CONFIG ====
OCR_JSON_FILE = "financial_lines_psm6.json"

# Keywords to detect start of sections
keywords = {
    "balance_sheet": ["balance sheet", "statement of financial position"],
    "income_statement": [
        "income statement",
        "statement of profit or loss",
        "statement of comprehensive income",
    ]
}

# ==== LOAD JSON ====
with open(OCR_JSON_FILE, "r", encoding="utf-8") as f:
    ocr_lines = json.load(f)


# ==== HELPERS ====
def clean_text(text):
    """Remove non-printable characters and extra spaces."""
    if not isinstance(text, str):
        return ""
    text = re.sub(r"[^\x20-\x7E]", "", text)  # Remove non-ASCII
    text = re.sub(r"\s+", " ", text).strip()
    return text


def extract_text(entry):
    """Extract raw text from OCR line entry (string or dict)."""
    if isinstance(entry, dict):
        return entry.get("text", "")
    return str(entry)


def looks_like_table_row(text):
    """
    Detect if a line looks like a financial statement row:
    Some text + at least one number.
    """
    # Example: "Cash and cash equivalents  1,250,000  950,000"
    return bool(re.search(r"[A-Za-z]", text) and re.search(r"\d", text))


def find_section(keyword_list):
    """Find section of OCR text starting from any keyword."""
    section_lines = []
    capturing = False

    for entry in ocr_lines:
        text_val = extract_text(entry)
        text = clean_text(text_val)
        lower_text = text.lower()

        if any(k in lower_text for k in keyword_list):
            capturing = True
            continue

        if capturing:
            if text == "" or "notes" in lower_text or (
                "statement" in lower_text and not any(k in lower_text for k in keyword_list)
            ):
                break
            section_lines.append(text)

    return section_lines


def extract_table_rows(section_lines):
    """Extract only numeric table-like rows from section."""
    table_rows = []
    for line in section_lines:
        if looks_like_table_row(line):
            # Split into parts: last 2 or 3 parts may be numbers
            parts = line.split()
            # Try to guess numbers at the end
            num_parts = []
            text_parts = []
            for p in parts:
                if re.fullmatch(r"[\d,]+(\.\d+)?", p):
                    num_parts.append(p.replace(",", ""))
                else:
                    text_parts.append(p)

            table_rows.append({
                "label": " ".join(text_parts),
                "values": num_parts
            })
    return table_rows


# ==== MAIN PROCESS ====
results = {}
for name, keys in keywords.items():
    raw_section = find_section(keys)
    rows = extract_table_rows(raw_section)
    results[name] = rows

# ==== SHOW RESULTS ====
for section, rows in results.items():
    print(f"\n--- {section.upper()} ---")
    for row in rows[:10]:  # Show only first 10 for preview
        print(row)

# ==== SAVE RESULTS ====
with open("financial_tables_extracted.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)



--- BALANCE_SHEET ---

--- INCOME_STATEMENT ---


In [6]:
import re

def parse_table_row(line):
    # Clean line first
    line = line.strip()
    # Match the item name, optional note, and two numbers
    pattern = r"^(.*?)(?:\s+(\d[\d\.]*))?\s+([\d,()\-]+)\s+([\d,()\-]+)$"
    match = re.match(pattern, line)
    if not match:
        return None

    item, note, current, previous = match.groups()

    def parse_number(s):
        s = s.replace(",", "")
        if "(" in s and ")" in s:
            return -float(s.replace("(", "").replace(")", ""))
        return float(s)

    return {
        "item": item.strip(),
        "note": note.strip() if note else None,
        "current_year": parse_number(current),
        "previous_year": parse_number(previous)
    }

# Example
row_text = "Property, plant and equipment 12.1 739,976,584 261,560,140"
parsed = parse_table_row(row_text)
print(parsed)


{'item': 'Property, plant and equipment', 'note': '12.1', 'current_year': 739976584.0, 'previous_year': 261560140.0}


In [9]:
import re
import json

# === Example OCR lines ===
ocr_lines = [
    "--- BALANCE_SHEET ---",
    "Cash and cash equivalents - -",
    "Property, plant and equipment 12.1 739,976,584 261,560,140",
    "--- INCOME_STATEMENT ---",
    "Revenue 45,000,000 42,500,000",
    "Net profit 5,000,000 4,200,000"
]

# === Clean text function ===
def clean_text(text):
    text = re.sub(r"[^\x20-\x7E]", "", text)  # remove non-printable
    text = re.sub(r"\s+", " ", text).strip()
    return text

# === Find section lines by section name ===
def find_section(section_name):
    section_lines = []
    capturing = False
    for line in ocr_lines:
        text = clean_text(line)
        if text.strip() == f"--- {section_name} ---":
            capturing = True
            continue
        if capturing:
            if text.startswith("---"):  # next section starts
                break
            section_lines.append(text)
    return section_lines

# === Parse table row ===
def parse_table_row(line):
    parts = line.split()
    
    # Assume last two parts are current and previous year
    if len(parts) < 3:
        return None  # not enough data
    
    item = " ".join(parts[:-3]) if len(parts) > 3 else parts[0]
    note = parts[-3] if len(parts) > 3 else None
    current = parts[-2]
    previous = parts[-1]
    
    def parse_number(s):
        s = s.replace(",", "")
        if s == "-" or s == "":
            return 0.0
        if "(" in s and ")" in s:
            return -float(s.replace("(", "").replace(")", ""))
        return float(s)
    
    return {
        "item": item.strip(),
        "note": note.strip() if note else None,
        "current_year": parse_number(current),
        "previous_year": parse_number(previous)
    }

# === Extract all rows from a section ===
def extract_table_rows(section_lines):
    rows = []
    for line in section_lines:
        row = parse_table_row(line)
        if row:
            rows.append(row)
    return rows

# === Sections to extract ===
sections = ["BALANCE_SHEET", "INCOME_STATEMENT"]

# === Extract data ===
results = {}
for section in sections:
    raw_section = find_section(section)
    rows = extract_table_rows(raw_section)
    results[section] = rows

# === Save results to JSON ===
json_output = json.dumps(results, indent=4)
print(json_output)


{
    "BALANCE_SHEET": [
        {
            "item": "Cash and cash",
            "note": "equivalents",
            "current_year": 0.0,
            "previous_year": 0.0
        },
        {
            "item": "Property, plant and equipment",
            "note": "12.1",
            "current_year": 739976584.0,
            "previous_year": 261560140.0
        }
    ],
    "INCOME_STATEMENT": [
        {
            "item": "Revenue",
            "note": null,
            "current_year": 45000000.0,
            "previous_year": 42500000.0
        },
        {
            "item": "Net",
            "note": "profit",
            "current_year": 5000000.0,
            "previous_year": 4200000.0
        }
    ]
}


In [11]:
import re
import json

# === Sample OCR lines (replace this with your actual OCR output) ===
ocr_lines = [
    "Cash and cash equivalents - -",
    "Property, plant and equipment 12.1 739,976,584 261,560,140",
    "Revenue - 45,000,000 42,500,000",
    "Net profit - 5,000,000 4,200,000"
]

# === Sections and their keywords ===
sections_keywords = {
    "BALANCE_SHEET": ["cash", "property", "equipment"],
    "INCOME_STATEMENT": ["revenue", "profit", "loss"]
}

# === Text cleaning function ===
def clean_text(text):
    text = re.sub(r"[^\x20-\x7E]", "", text)  # remove non-printable characters
    text = re.sub(r"\s+", " ", text).strip()
    return text

# === Find section by keywords ===
def find_section(keyword_list):
    section_lines = []
    for line in ocr_lines:
        text = clean_text(line)
        lower_text = text.lower()
        if any(k in lower_text for k in keyword_list):
            section_lines.append(text)
    return section_lines

# === Parse a single table row ===
def parse_table_row(line):
    parts = line.split()
    if len(parts) < 2:
        return None

    current = parts[-2]
    previous = parts[-1]

    # Detect optional note
    note_candidate = parts[-3] if len(parts) >= 3 else None
    if note_candidate and re.match(r"^[\d\.]+$", note_candidate):
        note = note_candidate
        item = " ".join(parts[:-3])
    else:
        note = None
        item = " ".join(parts[:-2])

    # Remove trailing hyphens or spaces from item
    item = item.rstrip(" -")

    # Convert number, handle empty or "-"
    def parse_number(s):
        s = s.replace(",", "")
        if s == "-" or s == "":
            return 0.0
        if "(" in s and ")" in s:
            return -float(s.replace("(", "").replace(")", ""))
        return float(s)

    return {
        "item": item.strip(),
        "note": note.strip() if note else None,
        "current_year": parse_number(current),
        "previous_year": parse_number(previous)
    }

# === Extract all table rows from a section ===
def extract_table_rows(section_lines):
    rows = []
    for line in section_lines:
        row = parse_table_row(line)
        if row:
            rows.append(row)
    return rows

# === Main processing ===
results = {}
for section, keywords in sections_keywords.items():
    raw_section = find_section(keywords)
    rows = extract_table_rows(raw_section)
    results[section] = rows

# === Save results to JSON ===
json_output = json.dumps(results, indent=4)
print(json_output)


{
    "BALANCE_SHEET": [
        {
            "item": "Cash and cash equivalents",
            "note": null,
            "current_year": 0.0,
            "previous_year": 0.0
        },
        {
            "item": "Property, plant and equipment",
            "note": "12.1",
            "current_year": 739976584.0,
            "previous_year": 261560140.0
        }
    ],
    "INCOME_STATEMENT": [
        {
            "item": "Revenue",
            "note": null,
            "current_year": 45000000.0,
            "previous_year": 42500000.0
        },
        {
            "item": "Net profit",
            "note": null,
            "current_year": 5000000.0,
            "previous_year": 4200000.0
        }
    ]
}
